# comparing bc models with or without actions in the states

In [50]:
import os
import h5py
import pandas as pd
import numpy as np
import json
from scipy.stats import wilcoxon

In [51]:
import sys
sys.path.insert(0, "/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")

from src.data.data_processor_for_shards import ShardProcessor

In [55]:
models_dir = "/home/jake/Documents/AI_Clinician/models/bc_transformer/choc/sepsis/intake-vis/"
with_actions_dir = os.path.join(models_dir, "with_actions", "final_model")
without_actions_dir = os.path.join(models_dir, "without_actions", "final_model")

random_search_folder_name = "choc_bc_transformer_action_standard_sepsis"

# create a list of all folders that begin with the random search folder name
random_search_folders = [f for f in os.listdir(models_dir) if f.startswith(random_search_folder_name)]

for folder in random_search_folders:
    # list the folders inside folder
    inner_folders = os.listdir(os.path.join(models_dir, folder))

    # load results.json from inner_folders[0]
    for inner_folder in inner_folders:
        if inner_folder == "config_manager.pkl":
            continue

        with open(os.path.join(models_dir, folder, inner_folder, "results.json"), "r") as f:
            results = json.load(f)
            actions_in_state = results['hyperparameters']['actions_in_state']
            print(f"Folder: {folder}, Actions in state: {actions_in_state}")




Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133210_659767_job2883197.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133210_659767_job2883197.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133210_659767_job2883197.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133210_659767_job2883197.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133210_659767_job2883197.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133443_614623_job2883207.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133443_614623_job2883207.pbs-7, Actions in state: False
Folder: choc_bc_transformer_action_standard_sepsis_split_0_20260529_133443_614623_job2883207.pbs-7, Actions in state: False
Folder: 

In [ ]:
actions = ['intake', 'vis']

models_dir = "/home/jake/Documents/AI_Clinician/models/bc_transformer/choc/sepsis/intake-vis/"
with_actions_dir = os.path.join(models_dir, "with_actions", "final_model")
without_actions_dir = os.path.join(models_dir, "without_actions", "final_model")

# find the folder inside with_actions_dir and without_actions_dir
with_actions_experiment_dir = os.path.join(with_actions_dir, os.listdir(with_actions_dir)[0])
without_actions_experiment_dir = os.path.join(without_actions_dir, os.listdir(without_actions_dir)[0])

predictions_file = "test_set_predictions.h5"

# load the predictions from both models
keys_to_load = ['stay_ids', 'index_ptr', 'predictions', 'targets', 'event']
with_actions_predictions_path = os.path.join(with_actions_experiment_dir, predictions_file)
with_actions_predictions =  ShardProcessor.load_dynamic_deephit_predictions_from_h5(
    with_actions_predictions_path, 
    keys_to_load=keys_to_load, 
    keys_to_exclude=None
)

without_actions_predictions_path = os.path.join(without_actions_experiment_dir, predictions_file)
without_actions_predictions =  ShardProcessor.load_dynamic_deephit_predictions_from_h5(
    without_actions_predictions_path,
    keys_to_load=keys_to_load,
    keys_to_exclude=None
)

In [28]:
# calculate squared error for each prediction, then perform Wilcoxon signed-rank test to compare the two models' predictions
with_actions_squared_error = (with_actions_predictions['predictions'] - with_actions_predictions['targets']) ** 2
without_actions_squared_error = (without_actions_predictions['predictions'] - without_actions_predictions['targets']) ** 2

In [33]:
# compare the two models using the Wilcoxon signed-rank test on each action in actions
for i, action in enumerate(actions):
    with_actions_error_for_action = with_actions_squared_error[:, i]
    without_actions_error_for_action = without_actions_squared_error[:, i]
    
    stat, p_value = wilcoxon(with_actions_error_for_action, without_actions_error_for_action)
    print(f"Action: {action}, Wilcoxon statistic: {stat}, p-value: {p_value}")

    # quantify the effect size using Cohen's d
    mean_diff = np.mean(with_actions_error_for_action - without_actions_error_for_action)
    pooled_std = np.sqrt((np.std(with_actions_error_for_action) ** 2 + np.std(without_actions_error_for_action) ** 2) / 2)
    cohens_d = mean_diff / pooled_std
    print(f"Action: {action}, Cohen's d: {cohens_d}")


Action: intake, Wilcoxon statistic: 7720636828.5, p-value: 0.0
Action: intake, Cohen's d: 0.018958159734483986
Action: vis, Wilcoxon statistic: 8416057835.5, p-value: 0.0
Action: vis, Cohen's d: 0.01185991688154371


(195410, 2)